# Memory and State: What Survives the Slide

You have probably already been burned by an agent that forgot.

Maybe you taught it something in the morning, a file location, a decision, a name, and by the afternoon session it stared at you blankly, because for it there was no morning. Or you watched the opposite failure: a conversation that ran long and got *worse* as it went, the early instructions buried under an avalanche of later turns, until the agent contradicted something it had been told at the start. One agent remembers nothing. The other remembers everything until it drowns. Yours, right now, manages to be both.

Week 2's research assistant is a brilliant amnesiac. It validates every request, cites its sources, recovers from failures, and at the end of every run it forgets all of it, because the prompt is its only memory and the prompt dies with the process. And inside a single run, the prompt is also a memory that only grows: on Day 5 you printed the full trace and watched every turn re-send the entire history. That cannot grow forever. Real models have finite context windows, and even before the wall, a swollen prompt costs money and attention.

**This week the agent gets a past.** Two kinds of memory, deliberately different: a *sliding window* that keeps the last few exchanges of the current session and lets the rest go, and a *fact store*, a small SQLite database where individual facts are written down with timestamps and survive any restart. Joining them is a *summarization step*: when the window slides, the expiring turns are summarized into the store first, so what matters crosses over before the rest is dropped.

**The one idea:** context windows end; memory is what survives. Memory is not a bigger window. It is an architecture: a fast, small, lossy working memory for now, a durable, timestamped, queryable store for later, and a judgment call, made over and over, about what is worth carrying from one to the other.

*Week 3 of the agent sprint, as a lesson. Week 2 is built and proven (the breakage log is written, the explain test is passed); this lesson is the concept behind the whole week, and Day 1, the sliding window, is the stair you are standing on.*

**What Week 3 is for:** feeling the difference between *seeing* and *keeping*. If, by Friday, you can point at a fact and say exactly how it survived, or exactly where one died, the week is working.

**How to read this notebook.** Same rhythm as Weeks 1 and 2, on purpose: definition first, then overview, then an example, then the mechanics. Every code cell has a lead-in (purpose, what to expect, why that output is your checkpoint) and its header comments are that lead-in compressed. The vocabulary carries over unchanged: prompt, Observation, registry, verdict. Week 1's sentence still governs everything: **an agent is a loop, and the model never *does* anything. It emits text. Your code acts on that text.** This week you will watch that sentence do new work: a summary is just text the model emits, until your code decides it deserves to become a fact.

## The Whole Thing, Once, Slowly: A Toy Walkthrough

Same ritual as Weeks 1 and 2: the entire week in one small story, before any code. This story needs two sessions, because memory is the thing that lives *between* sessions.

**Yesterday, Session 1.** You are setting up a test rig with the agent. The window for this story is small, the last 3 exchanges, so you can watch it work. The conversation goes:

> **Exchange 1.** You: "The staging box for Falcon tests is called Kestrel." Agent: "Noted."
> **Exchange 2.** You: "Morning. Coffee first, then we work." Agent: "Morning! Ready when you are."
> **Exchange 3.** You: "Default timeout in my fetch tool is 30 seconds." Agent: "Noted."
> **Exchange 4.** You: "Lunch is at noon today, usual place."

Exchange 4 does not fit. The window holds 3, so exchange 1, the Kestrel fact, is pushed out. But watch the new step: *before* it drops, your code hands the expiring exchange to the model with one instruction, "what's worth keeping from this?" The model emits text: `The staging box for Falcon tests is called Kestrel.` That text is just text, until your code acts on it: your code stores it in the fact store, as a row, with a timestamp and a source that says where it came from:

```
fact_text:  The staging box for Falcon tests is called Kestrel.
learned_at: 2026-10-19T08:14:03+00:00
source:     window summary, session 1
```

**Exchange 5** pushes out exchange 2, the coffee exchange. The summarizer gets its turn and keeps *nothing*, and nothing is stored. That empty answer is a feature: not every turn deserves to become permanent.

At the end of Session 1 you mention one more thing, the Falcon status page URL, and at write-back your code asks the model, "anything worth remembering?" and stores its reply the same way. Then the process ends. The window, all five exchanges of it, ceases to exist.

**Today, Session 2.** A brand-new process. The window is empty; the agent has seen nothing, ever. You ask:

> Where is the Falcon status page, and what is the staging box called?

Before the first model call, your code does the new thing: it queries the fact store with words from your question, finds two rows, and pastes them into the opening prompt under a heading, *Memory*. The model reads facts it never witnessed, in a session that did not exist yesterday, and answers both halves, citing the store. The staging box is Kestrel. It knows because yesterday's summarizer judged it worth keeping, and yesterday's code wrote it down.

Three different fates, one conversation: the Kestrel fact survived by being summarized; the coffee chat died on schedule and nobody mourns it; the status page survived by write-back. The window forgot everything, as designed. The store forgot nothing it was told to keep. The summarizer decided which was which.

Everything after this section is one piece of this story, taken slowly.

## Anatomy of a Remembering Agent

Week 2's turn had a gate in the middle of it. Week 3's change is not inside the turn at all. It is around the whole run:

| | Week 2 | Week 3 |
|---|---|---|
| At run start | Starts from zero, every time | Queries the fact store first; relevant facts are injected into the opening prompt |
| During a long run | The prompt grows until it breaks | The window keeps the last N exchanges; the rest is summarized or dropped |
| Across a restart | Total amnesia | Facts persist, each with a timestamp and a source |
| What decides what survives | Nothing. Everything is kept until it cannot be | The summarization step: a judgment call, made at every slide |
| The failure mode | Runs out of window | Remembers things that stopped being true (stale facts), or remembers two things that disagree |

The loop itself did not change. Ask, validate, dispatch, append the Observation, repeat. What changed is that the loop now has a past it can consult and a future it writes for. Keep the Week 2 picture in your head; this week hangs one new idea on it: **what survives is a decision, and your code is where the decision gets enforced.**

### Setup: two libraries this week

**Purpose of the cell below:** load this week's tools, all standard library, nothing to install. `sqlite3` is the fact store: a real database that lives in a single file and ships inside Python. `tempfile` gives the lesson a scratch folder, so the demo databases live outside your repo and every run of this notebook starts from a clean store. `datetime` provides the timestamps every fact will carry, and `json` is still here because the model's replies are still Week 2's JSON.

**What to expect:** two lines printed, the scratch folder and the fact store's file path inside it. Later cells, including a completely separate process, will need that exact path, which is why it is printed where you can see it. **Why this output is your checkpoint:** if the path points anywhere near your repo, something is wrong; the whole point is that this memory is disposable.

In [1]:
# Setup: sqlite3 for the fact store, tempfile for a scratch home for the
# demo databases, datetime for timestamps, json because the model's replies
# are still Week 2's JSON. subprocess and sys arrive now too: the restart
# test later needs to start a fresh Python process.
# The demo databases live in a fresh temp folder: re-running this notebook
# starts from an empty store, and nothing is left behind in the repo.
import json
import os
import re
import sqlite3
import subprocess
import sys
import tempfile
from datetime import datetime, timezone

SCRATCH_DIR = tempfile.mkdtemp(prefix="week03-lesson-")
DB_PATH = os.path.join(SCRATCH_DIR, "facts.db")
print("Scratch folder for this lesson:", SCRATCH_DIR)
print("Fact store file:              ", DB_PATH)

Scratch folder for this lesson: /tmp/week03-lesson-de9otxf6
Fact store file:               /tmp/week03-lesson-de9otxf6/facts.db


## The Sliding Window: Working Memory, On Purpose

**Definition.** A *sliding window* is a rule about what the agent is allowed to see: only the last N exchanges of the conversation. When a new exchange arrives and the window is already full, the oldest exchange leaves. The window itself archives nothing. Leaving the window means gone. Picture a whiteboard that fits N exchanges: to write a new one at the bottom, you erase the top line.

One word needs nailing down before anything is counted, because it is this week's classic off-by-one (your Day 1 sheet names it as the likely snag): an **exchange** is one user turn plus the agent's full response to it. In the wired loop that means the model's reply and the Observation it produced, together. Count exchanges, never individual messages, or your window will silently be half the size you think it is.

**Why it matters here.** Week 2's loop appends the reply and the Observation to the prompt, forever. You watched the consequence on Day 5 with the trace switch on: every turn's prompt contained the entire history, re-sent in full. Real context windows are finite, and the limit is not the only cost: a prompt bloated with last Tuesday's small talk makes the model wade for what matters. The window is the session's *working memory*: fast, always in view, and lossy on purpose. Its job is the current work, not the record.

In your build, the window starts at N=10 (your Day 1 sheet). The demo below uses N=3, not because 3 is right, but because the cut has to happen on screen where you can watch it.

**The cell below** builds the `SlidingWindow` and runs Monday's session through it: six scripted exchanges about the Falcon rig, the same cast as the walkthrough. One design decision to notice before you run it: `append()` *returns* whatever fell off. Today that return value is just printed. On Wednesday it becomes the summarizer's input, so the window is built from the start to hand over what it drops. *Expect:* the window filling at exchange 3, then one exchange falling off per new arrival, named as it goes: exchange 1, the Kestrel fact, falls at exchange 4; the coffee chat at 5; the timeout fact at 6. **Why this output is your checkpoint:** Day 1's done criteria are "the agent holds a conversation using only the window" and "you can point to the exact place the window cuts." This printout is that pointing, rehearsed: if you can say which append killed which fact, the mechanism is yours.

In [2]:
# SlidingWindow: the last N exchanges, and whatever falls off the end.
# append() returns the exchanges that expired, because on Wednesday those
# expired turns are exactly what the summarizer gets handed. A window that
# silently swallowed its drops could never feed a summarizer.
class SlidingWindow:
    """Working memory: holds the last max_exchanges exchanges, nothing more."""

    def __init__(self, max_exchanges):
        self.max_exchanges = max_exchanges
        self.exchanges = []

    def append(self, exchange):
        """Add one exchange; return the list of exchanges pushed out (0 or 1
        in practice, since exchanges arrive one at a time)."""
        self.exchanges.append(exchange)
        expired = []
        while len(self.exchanges) > self.max_exchanges:
            expired.append(self.exchanges.pop(0))
        return expired

    def get(self):
        """The exchanges the model is allowed to see right now."""
        return list(self.exchanges)


def show(window, label):
    print(label)
    for i, ex in enumerate(window.get(), 1):
        print(f"  [{i}] {ex['user']}  ->  {ex['agent']}")


# Monday's session, scripted: three facts worth keeping, two pieces of
# small talk, and one closing question. Watch which is which as they slide.
SESSION = [
    {"user": "The staging box for Falcon tests is called Kestrel.",
     "agent": "Noted: staging box is Kestrel."},
    {"user": "Morning. Coffee first, then we work.",
     "agent": "Morning! Ready when you are."},
    {"user": "Default timeout in my fetch tool is 30 seconds.",
     "agent": "Noted: fetch timeout defaults to 30 seconds."},
    {"user": "Lunch is at noon today, usual place.",
     "agent": "Got it: lunch at noon."},
    {"user": "The Falcon status page is https://falcon.example.com/status",
     "agent": "Noted: status page URL recorded."},
    {"user": "What is the staging box called?",
     "agent": "(about to answer from whatever it can still see)"},
]

window = SlidingWindow(max_exchanges=3)   # the build starts at 10; 3 so the cut is visible
for n, exchange in enumerate(SESSION, 1):
    expired = window.append(exchange)
    show(window, f"After exchange {n}:")
    for ex in expired:
        print(f"  FELL OFF: {ex['user']}")
    print()

print("Exchange 6 asks about the staging box. Exchange 1, which named it,")
print("left the window at exchange 4. Inside this session, it is already gone.")

After exchange 1:
  [1] The staging box for Falcon tests is called Kestrel.  ->  Noted: staging box is Kestrel.

After exchange 2:
  [1] The staging box for Falcon tests is called Kestrel.  ->  Noted: staging box is Kestrel.
  [2] Morning. Coffee first, then we work.  ->  Morning! Ready when you are.

After exchange 3:
  [1] The staging box for Falcon tests is called Kestrel.  ->  Noted: staging box is Kestrel.
  [2] Morning. Coffee first, then we work.  ->  Morning! Ready when you are.
  [3] Default timeout in my fetch tool is 30 seconds.  ->  Noted: fetch timeout defaults to 30 seconds.

After exchange 4:
  [1] Morning. Coffee first, then we work.  ->  Morning! Ready when you are.
  [2] Default timeout in my fetch tool is 30 seconds.  ->  Noted: fetch timeout defaults to 30 seconds.
  [3] Lunch is at noon today, usual place.  ->  Got it: lunch at noon.
  FELL OFF: The staging box for Falcon tests is called Kestrel.

After exchange 5:
  [1] Default timeout in my fetch tool is 30 secon

## The Fact Store: Memory That Survives the Process

**Definition.** A *fact store* is durable memory: a table where facts are written down as rows, deliberately, one fact per row. Each row carries three things: the fact itself as text, `learned_at`, a timestamp recording when it was learned, and `source`, where it came from. The store for this week is SQLite, which deserves its own plain definition: SQLite is a real database that lives in a single file and is built into Python's standard library. No server, no install, no account. If your code can write a file, it can keep a database.

**Overview.** The store has exactly two operations, and the week's discipline is in how few there are:

- `remember(fact, source)` writes one row. Writing is deliberate: nothing enters the store by accident, only when code decides a piece of text deserves to become durable.
- `recall(query)` reads rows back by keyword match: a fact is returned if the query text appears in it. That is dumb on purpose (your Day 2 sheet's snag, pre-empted): if you are tempted by embeddings and semantic search, that is parking-lot material. This week's lesson is the write/query discipline, not retrieval quality.

Two columns do quiet load-bearing work. The **timestamp** is what makes Friday's questions answerable at all: "which of these two facts is newer?" *is* the `learned_at` column, and stale memory is just a fact whose timestamp nobody checked. The **source** is Week 2's citation instinct applied to memory: a fact whose origin you cannot name is a rumor with a timestamp. One design choice, stated now so Friday is honest: `recall` returns matches *newest first*, so when two facts disagree, the freshest sits on top. Whether newest *should* win is Friday's policy question, not today's.

**The cell below** opens a fresh store at the scratch path from Setup (deleting any leftover file first, so re-runs are clean), creates the `facts` table, defines `remember` and `recall`, and seeds six facts by hand: real sprint facts, plus a deliberate contradiction pair about the Falcon dashboard, seeded with fixed timestamps a few days apart. *Expect:* six rows printed, each with its text, timestamp, and source, both dashboard facts present side by side. **Why this output is your checkpoint:** the schema *is* Tuesday's contract, printed in plain sight. Note what the store does with the contradiction: nothing. It records both, faithfully. Resolving them is not the store's job, and Friday you will find out whose job it is.

In [3]:
# The fact store: one table, two operations.
# facts(id, fact_text, learned_at, source). The timestamp and the source are
# not decoration: Friday's stale-fact and contradiction tests are answered
# from those two columns, so they are written on every single row.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)          # fresh store every run of the notebook

def open_store(path):
    """Open the store at path, creating the facts table if it is new."""
    conn = sqlite3.connect(path)
    conn.execute("""CREATE TABLE IF NOT EXISTS facts (
        id INTEGER PRIMARY KEY,
        fact_text TEXT NOT NULL,
        learned_at TEXT NOT NULL,
        source TEXT)""")
    return conn

def now_iso():
    """Current UTC time as an ISO string: sortable, readable, unambiguous."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def remember(conn, fact, source, learned_at=None):
    """Write one fact. learned_at defaults to now; a caller may pass a fixed
    timestamp so that ordering never depends on how fast code happens to run."""
    conn.execute(
        "INSERT INTO facts (fact_text, learned_at, source) VALUES (?, ?, ?)",
        (fact, learned_at or now_iso(), source))
    conn.commit()

def recall(conn, query):
    """Keyword match: facts containing the query text, newest first.
    Dumb on purpose: the lesson is the write/query discipline, not retrieval
    quality. Newest first means a contradiction surfaces freshest on top,
    which Friday's tests will have opinions about."""
    return conn.execute(
        "SELECT fact_text, learned_at, source FROM facts "
        "WHERE fact_text LIKE ? ORDER BY learned_at DESC",
        (f"%{query}%",)).fetchall()

# Seed by hand, the Day 2 way: real facts from the sprint so far, plus one
# deliberate contradiction pair about the Falcon dashboard, four days apart.
db = open_store(DB_PATH)
remember(db, "SAFE_DIR is the sandbox: week-02-tools-structured-output/sandbox",
         "Day 2 seed, by hand")
remember(db, "The timeout bug hid on the unhappy path, not the happy one",
         "Day 2 seed, by hand")
remember(db, "DeepSeek V4 Pro fenced its JSON in 7 of 10 replies on Day 4",
         "Day 2 seed, by hand")
remember(db, "The Falcon dashboard is not public yet",
         "seeded demo fact", learned_at="2026-10-20T09:00:00+00:00")
remember(db, "The Falcon dashboard went public",
         "seeded demo fact", learned_at="2026-10-24T09:00:00+00:00")
remember(db, "A citation you have not checked is a rumor",
         "Day 2 seed, by hand")

for row in db.execute("SELECT id, fact_text, learned_at, source FROM facts ORDER BY id"):
    print(row)

(1, 'SAFE_DIR is the sandbox: week-02-tools-structured-output/sandbox', '2026-10-30T12:24:43+00:00', 'Day 2 seed, by hand')
(2, 'The timeout bug hid on the unhappy path, not the happy one', '2026-10-30T12:24:43+00:00', 'Day 2 seed, by hand')
(3, 'DeepSeek V4 Pro fenced its JSON in 7 of 10 replies on Day 4', '2026-10-30T12:24:43+00:00', 'Day 2 seed, by hand')
(4, 'The Falcon dashboard is not public yet', '2026-10-20T09:00:00+00:00', 'seeded demo fact')
(5, 'The Falcon dashboard went public', '2026-10-24T09:00:00+00:00', 'seeded demo fact')
(6, 'A citation you have not checked is a rumor', '2026-10-30T12:24:43+00:00', 'Day 2 seed, by hand')


## The Restart: The Only Test That Counts

**Definition, courtesy of your Day 2 sheet:** memory that dies with the process isn't memory. The restart is not a formality; it is the entire difference between the window and the store. The window lives in the prompt, and the prompt is rebuilt by a running process. The store lives in a file, and files do not care which process opens them.

This needs an honest demonstration, and a notebook makes that awkward: the kernel running these cells is one process, and closing a database connection inside it proves little, because the same process could be remembering things in variables. So the cell below does the real version: it starts a **completely separate Python process**, hands it nothing but the store's file path, and lets that stranger open the file and recall on its own. The fresh process has never seen this notebook's variables, its `SESSION`, or its window. Everything it reports came off disk.

**The cell below** recalls two things through the stranger: `sandbox`, which should return the SAFE_DIR fact seeded above, and `Falcon dashboard`, which should return both halves of the contradiction, newest first. *Expect:* a block of output clearly marked as the fresh process's own, with the facts and their timestamps and sources attached. **Why this output is your checkpoint:** it is Tuesday's done criteria, "a fact written in one session, recalled across a full restart," demonstrated rather than asserted. When your real build does this on Tuesday, the restart will be you killing the agent and starting it again. The file will not notice the difference.

In [4]:
# The restart test, honestly: a brand-new Python process that has never
# seen this notebook's variables. It gets only the database file path,
# opens the file itself, and recalls. If the facts lived only in this
# process's memory, the stranger would find nothing.
FRESH_PROCESS = """
import sqlite3, sys
conn = sqlite3.connect(sys.argv[1])
for query in ["sandbox", "Falcon dashboard"]:
    print("recall(%r):" % query)
    for fact, learned_at, source in conn.execute(
        "SELECT fact_text, learned_at, source FROM facts "
        "WHERE fact_text LIKE ? ORDER BY learned_at DESC", ("%" + query + "%",)):
        print("  - %s  [%s, %s]" % (fact, learned_at, source))
"""

result = subprocess.run([sys.executable, "-c", FRESH_PROCESS, DB_PATH],
                        capture_output=True, text=True)
print("--- output of the fresh process ---")
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="")

--- output of the fresh process ---
recall('sandbox'):
  - SAFE_DIR is the sandbox: week-02-tools-structured-output/sandbox  [2026-10-30T12:24:43+00:00, Day 2 seed, by hand]
recall('Falcon dashboard'):
  - The Falcon dashboard went public  [2026-10-24T09:00:00+00:00, seeded demo fact]
  - The Falcon dashboard is not public yet  [2026-10-20T09:00:00+00:00, seeded demo fact]


## The Summarization Step: The Bridge Between the Tiers

**Definition.** The *summarization step* is what happens in the moment the window slides. The expiring exchanges are not dropped immediately. First, your code hands them to the model with one instruction, "what's worth keeping from this?", and whatever text the model emits back is stored in the fact store, via `remember()`, as a timestamped fact. Only then do the exchanges leave the window. Dropping is data loss. Summarizing first is a judgment call, and the call is made fresh at every slide.

Hold Week 1's sentence up against this, because it does real work here: **an agent is a loop, and the model never *does* anything. It emits text. Your code acts on that text.** The model does not reach into your database. It emits a summary, which is just text, and *your code* decides that text deserves to become a fact, stamps it with a timestamp and a source (`window summary, ...`, so future-you can tell a summarized fact from a hand-seeded one), and writes the row. The model proposes memory. Your code commits it.

**The walkthrough.** One expiry, taken slowly:

1. **It sees this:** the window, full, and a new exchange arriving. **Your code runs:** `window.append(new_exchange)`, which hands back the exchange that fell off. **This means:** the drop is no longer silent. The window was built (Monday) to return its expirations precisely so this step has something to catch.
2. **It sees this:** the expired exchange. **Your code runs:** the summarization prompt: the expiring turns plus the instruction. **This means:** the model reads material that is about to stop existing and answers a narrower question than "summarize this": not what was said, but what is *worth keeping*.
3. **It sees this:** the model's reply text. **Your code runs:** `remember(store, summary_text, source="window summary, ...")`, then lets the exchange go. **This means:** the fact now lives in the tier that survives, wearing a timestamp and a provenance. The wording, the jokes, and everything the summarizer judged unimportant do not. That loss is not a bug in the step; it *is* the step.

<div align="center">

An exchange expires from the window

↓

Your code asks the model: what's worth keeping from this?

↓

The model emits summary text (or nothing)

↓

Your code stores it via `remember()`, timestamped, sourced

↓

The exchange drops. The fact stays.

</div>

**Two disciplines from your Day 3 sheet, stated before you watch the demo.** First, the prompt decides the genre: ask loosely and the summarizer writes diary entries, narrative prose about a nice chat; the build's prompt demands *facts, one per line, no narrative*, because summaries are for retrieval, not for reading. Second, the summarizer's most powerful answer is *nothing*. Small talk, today's lunch, anything whose value expires with the day should produce no fact at all, and a summarizer that stores everything has quietly rebuilt the problem the window exists to solve, one tier down.

**The cell below** wires Monday's window to Tuesday's store through a mock summarizer and re-runs Monday's exact session. The summarizer is simulated, and that is labeled rather than hidden: `MockSummarizer` pops pre-written summary replies, one per expiry, including an empty reply for the coffee chat. Everything around it (the expiry, the ask, the `remember()` call, the drop) is the real wiring. A separate store file is used, so Tuesday's seeded store stays untouched for comparison. *Expect:* three expiries; the Kestrel exchange produces a stored fact, the coffee exchange produces the summarizer's nothing, and the timeout exchange produces a stored fact, after which the store's full contents print: two facts, each sourced as a window summary. **Why this output is your checkpoint:** compare it against Monday's run of the *same session*. There, the Kestrel fact fell off and was gone. Here it is sitting in a database, timestamped, after the session that created it ended. That difference, one bridge, is Wednesday's whole lesson.

In [5]:
# The summarization step: expired turns are summarized BEFORE they drop.
# The summarizer below is a mock: canned summary text, popped in order, so
# the demo is reproducible. In the real build the model writes the summary;
# the wiring around it is identical either way:
#   window expiry -> summary text -> remember() -> the turn may leave.
class MockSummarizer:
    """Deterministic stand-in: pre-written summaries, one per expiry.
    An empty string means 'nothing here is worth keeping', and the wiring
    honors it: no row is written."""

    def __init__(self, script):
        self.script = list(script)

    def __call__(self, expired_exchange):
        return self.script.pop(0) if self.script else ""


SUMMARY_DB = os.path.join(SCRATCH_DIR, "summary.db")   # separate file: Tuesday's store stays as it was
if os.path.exists(SUMMARY_DB):
    os.remove(SUMMARY_DB)
store = open_store(SUMMARY_DB)

summarizer = MockSummarizer([
    "The staging box for Falcon tests is called Kestrel.",  # exchange 1 expires
    "",                                                     # exchange 2: coffee small talk
    "Default fetch timeout is 30 seconds.",                 # exchange 3 expires
])

window = SlidingWindow(max_exchanges=3)
for n, exchange in enumerate(SESSION, 1):
    for ex in window.append(exchange):
        summary = summarizer(ex)
        if summary:
            remember(store, summary, source=f"window summary, exchange {n - 3}")
            print(f"exchange {n}: a turn expired -> summary stored: {summary}")
        else:
            print(f"exchange {n}: a turn expired -> summarizer kept nothing")

print()
print("The store now holds:")
for fact, learned_at, source in store.execute(
        "SELECT fact_text, learned_at, source FROM facts ORDER BY id"):
    print(f"  - {fact}  [{source}]")

exchange 4: a turn expired -> summary stored: The staging box for Falcon tests is called Kestrel.
exchange 5: a turn expired -> summarizer kept nothing
exchange 6: a turn expired -> summary stored: Default fetch timeout is 30 seconds.

The store now holds:
  - The staging box for Falcon tests is called Kestrel.  [window summary, exchange 1]
  - Default fetch timeout is 30 seconds.  [window summary, exchange 3]


## Two Tiers, One Agent

Quick grounding in the two memories side by side, because Thursday's wiring only makes sense if their jobs stay distinct:

| | Sliding window | Fact store |
|---|---|---|
| Lives in | The prompt, rebuilt every turn | A file on disk |
| Survives a restart | No. It dies with the process | Yes. Any process can reopen the file |
| Holds | The last N exchanges, in full | Individual facts, one row each |
| Loses | Everything older than N, silently | Detail: a summary keeps the fact, drops the wording and the context |
| The model reads it by | It is simply there, in view | Your code must query it and inject the results |
| Failure mode | Forgets the beginning | Remembers what stopped being true |

**Why not just a bigger window?** This is the explain test's favorite trap, so meet it here first. A bigger window *delays* the cut; it does not remove it, and a session can always be made longer than any window you can afford. The window is mortal at every size: restart the process and a million-exchange window is exactly as empty as a three-exchange one. And presence is not the same as retrieval: a fact buried in exchange 4,000 of a giant prompt is a needle the model may never attend to, while a fact your code *queried for* arrives at the top of the prompt, labeled, relevant to the question in hand. Size is a budget. Architecture is a decision. This week is the decision.

Notice, too, who operates the boundary between the tiers. When the model wants something kept, it cannot keep it. It emits text, a summary, a `{"remember": ...}` reply, and waits. **The model requests. Your code calls** `remember()`, or doesn't. Every durable fact in the store exists because code, acting on a model's text, chose to write it. That asymmetry is the same one from Week 1, wearing a new uniform, and it is why a bad fact in the store is ultimately a code-and-judgment problem, never a "the model remembered wrong" mystery.

## The Loop, With Memory

**Definition.** The *wired loop* is Week 2's research-assistant loop with three memory touches added, and nothing else changed. The contract is still JSON, the gate is still the validator, the tools are still dispatched by your code. Around that unchanged core:

1. **Recall at startup.** Before the first model call, your code pulls keywords from the question, queries the fact store, and pastes the matching facts into the opening prompt under a *Memory* heading. The model begins the run knowing what previous sessions learned, labeled as memory rather than as current conversation.
2. **The window during the run.** Exchanges collect in the sliding window instead of an ever-growing prompt. The prompt is assembled from the window's contents. If the window slides mid-run, Wednesday's summarizer gets the expirations before they drop.
3. **Write-back at the end.** When the run finishes, your code asks the model one more question, "anything worth remembering?", and stores the reply via `remember()`. A fact learned in this run becomes recallable in every later run. This is the touch that closes the circuit: the store is not just read by the loop, it is *written* by it.

**The cell below** runs the whole circuit twice, as two sessions against one fresh store file. Session 1 teaches the agent where the Falcon status page lives; watch for the write-back line at its end, because that single `remember()` call is the only reason Session 2 can succeed. Then a divider: a fresh window and a fresh model, the honest in-notebook version of a restart (Tuesday's separate process already proved the file itself survives anything). Session 2 asks a two-part question: what the README says Project Falcon is, answerable by a tool, and where its status page is, answerable *only* from Session 1's fact: no tool in Session 2 touches the web. *Expect:* Session 2's startup recall to print exactly one fact, the status page, sourced as a write-back; the `read_file` call to run as in Week 2; and a final answer whose citations name both the file and the fact store. **Why this output is your checkpoint:** it is Thursday's done criteria, "the agent answers a question using a fact from a previous session, end to end," rehearsed against the mock before you wire the real model in.

> **Simulated model ahead.** The next cell uses `MockModel`, the same stand-in as Weeks 1 and 2: pre-written replies, popped in order, including the write-back replies. The tools are simulated too (the Week 2 sandbox, one file). Everything the memory system does (recall injection, window bookkeeping, write-back, the store itself) is the real logic from this lesson's earlier cells. The mock replies arrive well-formed, so the loop below dispatches them directly; in your real build, Week 2's validator still gates every reply before dispatch, unchanged. Only the sources of text are fake, which is what makes the demo reproducible.

In [6]:
# The wired loop: Week 2's research assistant with three memory touches.
#   1. recall at startup  -- matching facts injected into the opening prompt
#   2. window during      -- exchanges collect in the window, not a raw history
#   3. write-back at end  -- the model's "worth remembering" text is stored
# The model still only emits text. Your code calls the tools, queries the
# store, and calls remember(). The model requests. Your code calls.
class MockModel:
    """Deterministic stand-in for ask(): pops canned replies in order.
    The loop cannot tell the difference -- it only ever sees text."""

    def __init__(self, script):
        self.script = list(script)
        self.calls = 0

    def __call__(self, prompt):
        self.calls += 1
        if not self.script:
            return '{"answer": "I am out of scripted replies.", "citations": []}'
        return self.script.pop(0)


# --- the mock tool: Week 2's sandbox, one file ------------------------------
SANDBOX_FILES = {
    "README.md": "Project Falcon is a tiny data logger for greenhouse sensors. "
                 "It records temperature and humidity every minute. "
                 "See the project page for status.",
}

def mock_read_file(rel_path):
    if rel_path in SANDBOX_FILES:
        return {"ok": True, "path": rel_path, "text": SANDBOX_FILES[rel_path]}
    return {"ok": False, "error": "not_found", "requested": rel_path}

MOCK_TOOLS = {"read_file": mock_read_file}

def observation_from(result):
    """Wrap a tool result dict as the Observation text the model reads."""
    if result["ok"]:
        return f"Observation: {result.get('text') or result.get('stdout') or ''}"
    return f"Observation: tool error ({result['error']}): {result.get('detail', '')}"


WEEK3_PROMPT = """You are a research agent with a memory. Facts under Memory were
learned in earlier sessions; treat them as true unless this session's tools
contradict them.
Memory:
{facts}
Reply with ONLY a JSON object, no other text. To use a tool:
{"tool": "<tool name>", "arguments": { ... }}
Available tools: read_file (arguments: rel_path).
When you have enough to answer, reply with ONLY:
{"answer": "<your answer>", "citations": ["<file or fact store>", ...]}

Question: {question}
"""

STOPWORDS = {"what", "does", "the", "is", "a", "an", "and", "where", "its",
             "it", "to", "of", "for", "this", "that", "say", "says"}

def recall_for_question(conn, question):
    """Startup recall: pull keywords out of the question, recall each one,
    dedupe, keep newest first. Crude, and honestly labeled: keyword recall
    misses anything the question does not name. Thursday's snag, on purpose."""
    words = re.findall(r"[a-z0-9]+", question.lower())
    keywords = [w for w in dict.fromkeys(words) if w not in STOPWORDS]
    facts, seen = [], set()
    for kw in keywords:
        for row in recall(conn, kw):
            if row[0] not in seen:
                seen.add(row[0])
                facts.append(row)
    return facts


def run_wired(question, model, conn, window, max_iterations=10):
    """The Week 3 loop: recall, run on the window, write back at the end."""
    # Touch 1: recall at startup, injected into the opening prompt.
    facts = recall_for_question(conn, question)
    print("Recalled at startup:")
    if facts:
        for fact, learned_at, source in facts:
            print(f"  - {fact}  [{source}]")
    else:
        print("  (nothing: the store has no facts matching this question)")
    facts_block = "\n".join(f"- {f[0]}" for f in facts) or "(empty)"

    history = ""
    answer = None
    for turn in range(1, max_iterations + 1):
        # .replace, not .format: the template is full of literal JSON braces.
        prompt = (WEEK3_PROMPT.replace("{facts}", facts_block)
                              .replace("{question}", question) + history)
        reply = model(prompt)
        print(f"--- Turn {turn}: model reply ---")
        print(reply)
        try:
            parsed = json.loads(reply)
        except json.JSONDecodeError:
            parsed = None
        if isinstance(parsed, dict) and "answer" in parsed:
            answer = parsed
            window.append({"user": reply, "agent": "(final answer)"})
            print("Answer accepted: shape valid.")
            break
        # A tool call. In the real build, Week 2's validate_tool_call gates
        # this reply before dispatch; the mock's replies arrive well-formed.
        verdict = parsed
        fn = MOCK_TOOLS[verdict["tool"]]
        obs = observation_from(fn(**verdict["arguments"]))
        print(obs)
        window.append({"user": reply, "agent": obs})   # Touch 2: the window keeps the run
        history += "\n" + reply + "\n" + obs + "\n"

    # Touch 3: write-back. One more model call, a different question:
    # not "what happened?" but "what is worth keeping?"
    wb = json.loads(model('Anything worth remembering from this run? '
                          'Reply with ONLY {"remember": "<one fact>"} or {"remember": null}.'))
    if wb.get("remember"):
        remember(conn, wb["remember"], source="write-back, end of run")
        print(f"Write-back: stored '{wb['remember']}'")
    else:
        print("Write-back: nothing new stored.")
    return answer


WIRED_DB = os.path.join(SCRATCH_DIR, "wired.db")
if os.path.exists(WIRED_DB):
    os.remove(WIRED_DB)
wired = open_store(WIRED_DB)

print("===== Session 1: teaching the agent a fact =====")
session1 = MockModel([
    '{"answer": "Noted: the Falcon status page is https://falcon.example.com/status.", "citations": []}',
    '{"remember": "The Falcon status page is https://falcon.example.com/status"}',
])
run_wired("Note this for later: the Falcon status page is "
          "https://falcon.example.com/status",
          session1, wired, SlidingWindow(max_exchanges=3))

print()
print("===== Session 2: fresh window, fresh model, same store file =====")
print("(The process 'restarted'. Only the file carried over.)")
session2 = MockModel([
    '{"tool": "read_file", "arguments": {"rel_path": "README.md"}}',
    '{"answer": "Project Falcon is a tiny data logger for greenhouse sensors, '
    'recording temperature and humidity every minute. Its status page is '
    'https://falcon.example.com/status.", "citations": ["README.md", "fact store"]}',
    '{"remember": null}',
])
run_wired("What does the README say Project Falcon is, and where is its status page?",
          session2, wired, SlidingWindow(max_exchanges=3))

===== Session 1: teaching the agent a fact =====
Recalled at startup:
  (nothing: the store has no facts matching this question)
--- Turn 1: model reply ---
{"answer": "Noted: the Falcon status page is https://falcon.example.com/status.", "citations": []}
Answer accepted: shape valid.
Write-back: stored 'The Falcon status page is https://falcon.example.com/status'

===== Session 2: fresh window, fresh model, same store file =====
(The process 'restarted'. Only the file carried over.)
Recalled at startup:
  - The Falcon status page is https://falcon.example.com/status  [write-back, end of run]
--- Turn 1: model reply ---
{"tool": "read_file", "arguments": {"rel_path": "README.md"}}
Observation: Project Falcon is a tiny data logger for greenhouse sensors. It records temperature and humidity every minute. See the project page for status.
--- Turn 2: model reply ---
{"answer": "Project Falcon is a tiny data logger for greenhouse sensors, recording temperature and humidity every minute. Its

{'answer': 'Project Falcon is a tiny data logger for greenhouse sensors, recording temperature and humidity every minute. Its status page is https://falcon.example.com/status.',
 'citations': ['README.md', 'fact store']}

## Going Live: Your Build

The demo above ran on a simulated model, a simulated tool, and databases in a temp folder built to be thrown away. Your build runs the same architecture against your real model and a store file you keep. What changes:

- `MockModel` becomes your `ask_gateway(prompt)`: text in, text out, the same swap as Weeks 1 and 2. The loop cannot tell the difference.
- `MockSummarizer` becomes a real model call, carrying the real summarization prompt: the expiring turns plus "list the facts worth keeping, one per line, no narrative."
- The scratch database becomes `facts.db` sitting next to your agent in `week-03-memory/`, where it persists between runs, between days, between weeks. Tuesday's restart test stops being a demo and becomes your actual morning: kill the process, start it, ask.

### Where each piece lives in your repo

| Notebook section | Your file | Day |
|---|---|---|
| `SlidingWindow` | `week-03-memory/memory.py` | Day 1 |
| The fact store: schema, `remember`, `recall` | `week-03-memory/fact_store.py` | Day 2 |
| `summarize()`: the bridge step | `week-03-memory/memory.py` | Day 3 |
| The wired loop | your Week 2 research assistant, memory wired in | Day 4 |
| The three seam tests | `BREAKAGE_LOG.md` | Day 5 |

### Spot the difference (exercise)

Three things differ between the notebook's demo and your real build. Narrate each one out loud:

1. **The summarizer's judgment is real now.** The notebook's mock kept exactly what its script told it to keep. Your real model will keep things you would have dropped and drop things you would have kept. What does your prompt have to say to keep summaries factual rather than diary-shaped, and whose job is it to audit the difference on Wednesday?
2. **Keyword recall only finds what the question names.** The notebook's Session 2 worked because the question contained the words "Falcon" and "status page." A question like "what did we decide about the dashboard?" shares almost no words with the stored fact it needs. What does the agent do then, and why is that an acceptable loss this week rather than a bug?
3. **The store outlives the lesson.** The notebook's databases die with the temp folder. Yours accumulates: every write-back, every summary, every seed, for weeks. What does the `learned_at` column buy you in week six that it did not need to buy in this demo, and what happens to recall quality as the store grows?

## Your Build, Day by Day

| Day | You build | In file | Done when |
|---|---|---|---|
| 1 | `SlidingWindow`: last N exchanges, drops named as they fall | `memory.py` | The agent holds a multi-turn conversation using only the window, and you can point to the exact place the window cuts and name what was lost |
| 2 | The fact store: schema, `remember`, `recall` | `fact_store.py` | A fact written today is recalled on demand across a full process restart, and you can explain what each column is for |
| 3 | `summarize()`: expired turns become timestamped facts before dropping | `memory.py` | A long session slides the window and the important content lands in the store, and you can name one thing the summarizer kept, one it dropped, and whether you agree with both calls |
| 4 | The wired research assistant: recall at start, window during, write-back at end | your Week 2 loop + `week-03-memory/` | The agent answers a question using a fact from a previous session, live, end to end |
| 5 | Prove it: stress the seams, write it all up | `BREAKAGE_LOG.md` | A previous session's fact is recalled, the compression test is explained (what was summarized away, what was kept, and why), and all three experiments are written up with outcomes |

**Day 1, the stair you are on.** The day sheet's route: type the window first, under 30 lines, an `append` and a `get()`; then one small wiring chunk, the prompt built from the window instead of the full history; then the real work, which is watching: run the conversation, find the exact turn where early context falls off, and write down what the agent forgot. Commit: `week-03 day-01: sliding window short-term memory`.

**Day 2** is small code and a big test: schema first, then `remember`, then `recall`, seed 5 to 10 facts by hand, and then kill the process. The restart is the test. Commit: `week-03 day-02: SQLite fact store with timestamps`.

**Day 3** is the interesting day, and the sheet knows it: type the summarization prompt short and specific, wire it to the slide, then *judge* what it produces. The breakage log entry is a disagreement log: what the summarizer kept versus what you would have kept.

**Day 4** is plumbing, no new concepts: the startup query, the write-back, and one live run where a fresh session answers from an old fact. The sheet warns the write-back will store trivia. That is allowed today; Friday is when you find out why it matters.

**Day 5** is three experiments and three write-ups: a fact that becomes wrong, two facts that contradict, and a session long enough to summarize twice. The write-up is the deliverable as much as the runs.

## TODO: Your Hand-Typed Build

The notebook above is the worked example: read it, run it, argue with it. The build below is typed by hand, in your repo, one day at a time. Same rule as always: you type it, you run it, the done signal is binary.

**TODO, Day 1: the sliding window**
- **File:** `memory.py`, in folder `week-03-memory/`
- **Hand-type:** the `SlidingWindow` class: an `append` that returns what expired, and a `get()` returning the last N exchanges. Under 30 lines, comments in your own words. Then wire it into your loop so the prompt is built from the window.
- **Done signal:** a multi-turn conversation runs on the window alone, and you can point at the exact turn where the first exchange fell off and name what was lost.
- **Watch out:** define "exchange" before you count anything: one user turn plus the agent's full response. Count messages instead and your window is half the size you think it is.

**TODO, Day 2: the fact store**
- **File:** `fact_store.py`, in folder `week-03-memory/`
- **Hand-type:** the schema `facts(id, fact_text, learned_at, source)`, then `remember(fact, source)`, then `recall(query)`. Seed it by hand with 5 to 10 real facts from your sprint.
- **Done signal:** kill the process, start completely fresh, and a fact you wrote is recalled on demand. No restart, no credit.
- **Watch out:** recall is keyword match, dumb on purpose. The urge to reach for embeddings is parking-lot material; the discipline being built is deliberate writes and timestamped reads.

**TODO, Day 3: the summarization step**
- **File:** add `summarize()` to `memory.py`, in folder `week-03-memory/`
- **Hand-type:** the summarization prompt (short, specific: facts worth keeping, one per line, no narrative) and the `summarize(expired_turns)` step that fires when the window slides and stores its result via `remember()`.
- **Done signal:** a long session slides the window, and a summary sits in the store as a timestamped fact, which you have read back and judged: one keep you agree with, one drop you can defend or dispute.
- **Watch out:** if the summaries read like diary entries, the prompt is too loose. Summaries are for retrieval, not for reading.

**TODO, Day 4: wire memory into the agent**
- **File:** your Week 2 research assistant, importing from `week-03-memory/` (`memory.py`, `fact_store.py`)
- **Hand-type:** the startup recall (query the store with the question's keywords, inject the matches into the opening prompt) and the end-of-run write-back ("anything worth remembering?", stored via `remember()`). The window you already have from Day 1 runs during.
- **Done signal:** teach it a fact in one session, start a completely fresh session, and it answers a question that needs that fact. Live, end to end, once.
- **Watch out:** the write-back will happily store trivia. Let it, today. Friday's stale-fact test is where indiscriminate writing sends its bill.

**TODO, Day 5: stress the seams**
- **File:** `BREAKAGE_LOG.md`, in folder `week-03-memory/`
- **Hand-type:** the three experiment write-ups: the stale fact (taught X, then not-X), the contradiction (two sources, different timestamps), and the double compression (a session long enough to summarize twice, survivors compared round by round).
- **Done signal:** all three outcomes are written down, the agent has recalled a previous session's fact, and for the compression test you can say exactly what was summarized away, what was kept, and why.
- **Watch out:** there is no clean answer for contradictions, and that is the point. Timestamp-wins and source-wins are both defensible; the deliverable is the documented decision, not the right answer.

## Reference Code: The Build, File by File

The Week 3 build as files, the way it lives (or will live) in your repo. Reference, not a copy source: read for shape, type your own. Day 1 and Day 3 share a file, `memory.py`, because the summarizer exists to serve the window; Day 2 is the store; Day 4 is your Week 2 loop with the three touches marked.

A naming bridge, so nothing jars: the notebook's lesson code (functions taking an explicit connection, a window class, a mock summarizer) and the files below are the same code wearing repo clothes. In the files, the store's path is fixed beside the agent instead of passed around, and the summarizer takes the real model callable. The logic does not change; only the plumbing address does.

### Reference: `memory.py` (Days 1 and 3)

**Walkthrough.** Two residents, one file. `SlidingWindow` is Day 1: the working memory, holding exchanges and returning whatever each append pushes out. `summarize` is Day 3: it takes a batch of expired turns, formats them as text, calls the model with the summarization prompt, and stores every line the model returns via the fact store's `remember`, each stamped with the source `window summary`. Lines are the unit: the prompt demands facts one per line precisely so this function can store them one per row without parsing anything cleverer. An empty reply stores nothing, by design.

```python
"""Days 1 and 3 - the sliding window, and the summarization step that
catches what falls out of it.

The window is the session's working memory: the last N exchanges, nothing
older. The summarizer is the bridge to long-term memory: before an expired
turn drops, the model is asked what is worth keeping, and its answer is
stored in the fact store as timestamped facts. The model emits text.
This module's code does the storing.
"""

from fact_store import remember

# An exchange is one user turn plus the agent's full response (the model's
# reply and the Observation it produced). Count exchanges, never messages.
class SlidingWindow:
    """Holds the last max_exchanges exchanges; append() returns expirations."""

    def __init__(self, max_exchanges=10):
        self.max_exchanges = max_exchanges
        self.exchanges = []

    def append(self, exchange):
        """Add one exchange; return the list of exchanges pushed out."""
        self.exchanges.append(exchange)
        expired = []
        while len(self.exchanges) > self.max_exchanges:
            expired.append(self.exchanges.pop(0))
        return expired

    def get(self):
        """The exchanges the model is allowed to see right now."""
        return list(self.exchanges)


# The prompt decides the genre: facts, not diary. One fact per line so the
# lines can be stored one per row, unparsed.
SUMMARY_PROMPT = """Here are conversation turns that are about to be forgotten:

{turns}

List the facts worth keeping, one per line, no narrative.
If nothing is worth keeping, reply with nothing."""


def _turns_as_text(expired_turns):
    return "\n".join(
        f"User: {t['user']}\nAgent: {t['agent']}" for t in expired_turns)


def summarize(expired_turns, model, conn):
    """Summarize expired turns into the fact store. Returns the facts stored.

    model is the model callable (text in, text out). Its reply is just
    text until remember() commits each line as a fact, timestamped, with
    source 'window summary'. An empty reply stores nothing, on purpose.
    """
    if not expired_turns:
        return []
    reply = model(SUMMARY_PROMPT.replace("{turns}", _turns_as_text(expired_turns)))
    stored = []
    for line in reply.splitlines():
        line = line.strip()
        if line:
            remember(conn, line, source="window summary")
            stored.append(line)
    return stored
```

### Reference: `fact_store.py` (Day 2)

**Walkthrough.** The long-term store as a file that stands alone: it imports nothing from the rest of the build, so anything (the loop, a test script, a future week's planner) can open the same memory. `DB_PATH` is fixed beside this file, which is what "the store survives the process" means in practice: the file has an address, and every process that knows the address shares one past. `remember` and `recall` are the lesson's two operations, unchanged. The `__main__` block is the file's self-test, in the pattern of `structured.py`'s: seed two facts, recall one back, print what came home.

```python
"""Day 2 - the fact store: durable, timestamped, queryable memory.

A fact is a row: what it says, when it was learned, where it came from.
SQLite, standard library, one file on disk next to the agent. Two
operations only: remember() writes a fact, recall() reads facts back by
keyword, newest first. Deliberately dumb retrieval: the discipline is
deliberate writes and honest timestamps, not search quality.
"""

import os
import sqlite3
from datetime import datetime, timezone

DB_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "facts.db")


def connect(path=DB_PATH):
    """Open the store, creating the facts table if this is a new file."""
    conn = sqlite3.connect(path)
    conn.execute("""CREATE TABLE IF NOT EXISTS facts (
        id INTEGER PRIMARY KEY,
        fact_text TEXT NOT NULL,
        learned_at TEXT NOT NULL,
        source TEXT)""")
    return conn


def _now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def remember(conn, fact, source, learned_at=None):
    """Write one fact. The timestamp defaults to now and is never optional
    in the table: a fact without a when cannot take part in Friday's
    stale-fact questions at all."""
    conn.execute(
        "INSERT INTO facts (fact_text, learned_at, source) VALUES (?, ?, ?)",
        (fact, learned_at or _now_iso(), source))
    conn.commit()


def recall(conn, query):
    """Facts containing the query text, newest first. When two facts
    contradict, the freshest is returned on top; whether it should win
    is a policy the caller decides, not one the store smuggles in."""
    return conn.execute(
        "SELECT fact_text, learned_at, source FROM facts "
        "WHERE fact_text LIKE ? ORDER BY learned_at DESC",
        (f"%{query}%",)).fetchall()


if __name__ == "__main__":
    # Self-test: write in this process, read back in the same breath.
    # The real test is tomorrow's: a different process, same file.
    db = connect()
    remember(db, "SAFE_DIR is the sandbox", "fact_store self-test")
    remember(db, "The restart is the test: memory that dies with the "
                 "process isn't memory", "fact_store self-test")
    for fact, learned_at, source in recall(db, "sandbox"):
        print(f"{fact}  [{learned_at}, {source}]")
```

### Reference: the wired loop (Day 4)

**Walkthrough.** Your Week 2 research assistant, with the three Week 3 touches marked `# NEW (Week 3)`. Everything unmarked is the Week 2 build unchanged: the JSON contract, the validator gating every reply, the registry dispatching, citations checked at the end. Study the shape of the change, because it is the week's quiet thesis: memory did not require a new agent. It required three small, named insertion points in the loop you already owned: before the first call (recall), inside the turn cycle (the window owns the history now, and the summarizer catches its expirations), and after the last call (write-back). In Week 4, the planner-executor will read this same store as shared state; the loop that writes it stays this loop.

```python
"""Day 4 - the research assistant, with memory wired in.

Week 2's loop, three touches added:
  1. recall at startup: facts from previous sessions open the prompt
  2. the sliding window owns the history; expirations are summarized
     into the store before they drop
  3. write-back at the end: what the model says is worth remembering
     becomes next session's recall
The validator, the registry, and the citation discipline are Week 2's,
untouched. An agent is a loop; memory is what the loop consults and
what it leaves behind.
"""

import json

from ask_gateway import ask_gateway
from fact_store import connect, remember, recall
from memory import SlidingWindow, summarize
from structured import validate_tool_call, error_observation
from tools import TOOLS

MAX_ITERATIONS = 10

PROMPT = """You are a research agent with a memory. Facts under Memory were
learned in earlier sessions; treat them as true unless this session's
tools contradict them.
Memory:
{facts}
Reply with ONLY a JSON object, no other text. To use a tool:
{"tool": "<tool name>", "arguments": { ... }}
When you have enough to answer, reply with ONLY:
{"answer": "<your answer>", "citations": ["<file or URL>", ...]}

Question: {question}
"""

WRITE_BACK_PROMPT = ('Anything worth remembering from this run? '
                     'Reply with ONLY {"remember": "<one fact>"} '
                     'or {"remember": null}.')


def run(question, model=ask_gateway):
    conn = connect()
    window = SlidingWindow(max_exchanges=10)

    # NEW (Week 3), touch 1: recall at startup.
    facts = recall_for_question(conn, question)
    facts_block = "\n".join(f"- {f[0]}" for f in facts) or "(empty)"
    prompt = (PROMPT.replace("{facts}", facts_block)
                    .replace("{question}", question))

    for turn in range(1, MAX_ITERATIONS + 1):
        reply = model(prompt)
        try:
            parsed = json.loads(reply)
        except json.JSONDecodeError:
            parsed = None
        if isinstance(parsed, dict) and "answer" in parsed:
            if isinstance(parsed["answer"], str) and isinstance(parsed.get("citations"), list):
                answer = parsed
                break
            obs = ("Observation: invalid final answer: it must be a JSON "
                   "object with an 'answer' string and a 'citations' list.")
        else:
            verdict = validate_tool_call(reply)      # Week 2's gate, unchanged
            if not verdict["ok"]:
                obs = error_observation(verdict)
            else:
                fn = TOOLS[verdict["tool"]]
                obs = observation_from(fn(**verdict["arguments"]))

        # NEW (Week 3), touch 2: the window owns the history now, and
        # whatever slides out is summarized into the store first.
        for expired in window.append({"user": reply, "agent": obs}):
            summarize([expired], model, conn)
        prompt = build_prompt_from_window(window, facts_block, question)

    # NEW (Week 3), touch 3: write-back. The model requests; this code calls.
    wb = json.loads(model(WRITE_BACK_PROMPT))
    if wb.get("remember"):
        remember(conn, wb["remember"], source="write-back, end of run")
    return answer
```

Two helpers are left as exercises in the file's full version, both already built in this lesson's cells: `recall_for_question` (keyword extraction plus deduped recall, from Thursday's demo) and `build_prompt_from_window` (the prompt assembled from the window's exchanges rather than a raw ever-growing string). Neither hides a new idea; both are plumbing you have already watched work.

## Prove It: Mock vs Reality

The notebook proved the memory architecture against scripts: the window cut where arithmetic said it would, the store survived a stranger process, the summarizer kept what its script held. Now prove it against your real agent, where the summarizer's judgment is real and the facts fight back.

**1. Cross-session recall, live.** Your Day 4 build: teach it a fact in one session, kill the process completely, start a fresh session, and ask a question that needs the fact. Save the trace. If the answer names the fact and cites the store, Thursday's criteria are met in reality, not in rehearsal.

**2. The stale fact.** Teach it X ("the Falcon dashboard is not public yet"). Later, teach it not-X ("the Falcon dashboard went public"), properly timestamped. Ask about X. Record: did the agent notice the conflict at all? Which fact did the answer use, and can you explain the choice from the `learned_at` column and your recall ordering, or was it luck?

**3. The contradiction.** Seed two contradictory facts with *different* timestamps and *different* sources (a hand seed against a window summary is the honest version). Ask. Record which one the agent used and why you think it won: because it was newest, because of its source, or because of where it landed in the injected list. There is no clean right answer here, and that is the finding's shape: timestamp-wins and source-wins are both defensible policies. What is not defensible is having no policy and not knowing it.

**4. Double compression.** Run a session long enough to force two summarizations, so a fact must survive as: conversation, then summary, then material inside a later summary's input. Compare what survived round one against round two. Write down exactly what got summarized away, what was kept, and why you think the summarizer made those calls. This is Friday's third done criteria, and the one Week 6 will ask about again.

**5. Rehearse the explain test.** Out loud, no notes: (a) the sliding window versus the fact store, what each is for and what each loses; (b) why a bigger window does not solve the problem (your answer should reach all three: the cut still exists, the window still dies with the process, and presence is not retrieval); (c) your stale-fact outcome and your contradiction outcome from experiments 2 and 3, with the policy you are adopting and why.

The mock tells you the plumbing is correct. Reality tells you whether the judgment calls the plumbing carries, the summarizer's keeps and drops, the recall ordering, the write-back discipline, are calls you can defend. You need both before the week is done.

## Breakage Log: Mock vs Reality

_Fill this in Friday, in your own words. This becomes the week's `BREAKAGE_LOG.md`._

### Case 1: the stale fact
- **Mock did:**
- **Reality did:**
- **Finding:**

### Case 2: the contradiction
- **Seeded (fact, timestamp, source, both sides):**
- **Agent used:**
- **Why it won, and the policy I am adopting:**

### Case 3: double compression
- **Kept after round one:**
- **Kept after round two:**
- **Summarized away, and whether I agree:**

### Explain test rehearsal (out loud, no notes)
- [ ] window vs store: what each is for, what each loses
- [ ] why a bigger window doesn't solve it
- [ ] one stale outcome and one contradiction outcome, with my policy
- [ ] what my summarizer kept vs dropped on Wednesday, and why
- **Fuzzy spots to re-read:**

### Traces
_Paste the saved trace files from today's live runs here, or note their paths in the repo. Week 4's planner and workers will read this same fact store as shared state, and Week 6's eval harness will want these seam failures as test cases._

## What You Own Now

- Memory is an architecture, not a size. A bigger window delays the cut and still dies with the process; the two tiers exist because "recently in view" and "durably true" are different jobs.
- The window is working memory: fast, small, and lossy on purpose. The store is durable memory: timestamped, sourced, queryable, and exactly as trustworthy as the writes you allow into it.
- Summarization is a judgment call, made at every slide, by a model whose only power is emitting text. Your code commits the fact; your prompt shapes the judgment; your Friday policy decides the fights.
- Stale and contradictory memory are not edge cases. They are what memory *does* over time, and you have now met both in a lab, with timestamps, instead of meeting them in production without any.

Your Week 3 build was the exercise for this lesson: a window that names its losses, a store that survives its process, a summarizer whose keeps and drops you have personally audited, and a research assistant that starts each run knowing what the last one learned. If you can narrate a slide, a recall, a write-back, and a contradiction out loud right now, the notebook did its job.

**Where this goes:** Week 4 splits the one loop into a planner, workers, and a supervisor, and this fact store becomes the shared state they all read and write. The memory discipline you built this week, deliberate writes, honest timestamps, is what makes that sharing safe enough to try.